# Finding the Ground State Energy of a Molecule: the Theory

This kata introduces you to the problem of finding the ground state energy of a molecule.
This is the simplest example of a quantum chemistry problem that illustrates the workflow of solving such problems using quantum computing.

**This kata covers the following topics:**

- The problem of finding the ground state energy of a molecule and its connection to the phase estimation problem
- The workflow of obtaining the description of a molecule and converting it into inputs for a phase estimation routine
- Approximately implementing the unitary that acts as one of the inputs to phase estimation using Trotterization
- An end-to-end example of finding the ground state energy of $H_2$ molecule

> The exact science and tools involved in getting inputs to phase estimation is out of scope of this kata; it aims to give you a sense of overall workflow rather than go through each step involved in detail. For the purposes of code implementation we'll assume that the numeric parameters are already generated by earlier steps and provided as inputs.

**What you should know to start working on this kata:**

- Basics of quantum computing, including rotation gates
- Quantum phase estimation

This kata consists of two parts:
1. Part I (this notebook) walks you through the theory behind the approach we'll be taking, illustrated with a few Python demos.
2. [Part II](./GroundStateEnergyCode.ipynb) offers a series of Workbench problems and demos focused on implementing the described approach as a quantum program.

## Finding the ground state energy of a molecule: the problem

The _ground state energy_ of a quantum-mechanical system is defined as the lowest possible energy level this system can have. The corresponding state of the system is called its _ground state_, and it is the most stable configuration of the system. Thus, the ground state energy is useful for understanding other properties of the system and its behaviors.

> For example, the speed of a chemical reaction depends on the energies of reactants and products. Thus, estimating ground state energies of different molecules involved in a reaction is important for predicting its behavior. This, in turn, allows us to design and optimize complex multistep reactions involving catalysts which are hard to come up with experimentally.

## Connecting ground state energy with phase estimation problem

The evolution of a quantum-mechanical system over time is described by the _Schrödinger equation_:

$$i \frac{d}{dt} \ket{\Psi} = \hat{H} \ket{\Psi}$$

Here $\ket{\Psi}$ is the wave function that describes the state of the system at every moment of time, and $\hat{H}$ is the _Hamiltonian_ of the system - an operator that describes the total energy (potential and kinetic) of the system.

> Importantly, the canonical representations of the wave function and the Hamiltonian used in the Schrödinger equation (the ones you'll write down when considering the physics of the system) are different from the ones we'll use when writing our quantum code. In the canonical representation, the wave function is a function of time, spatial position or momentum of the particles that comprise the system, and other variables such as spin. Similarly, the potential and kinetic components of the Hamiltonian are defined in terms of particles, their physical properties such as mass, and forces acting between them.
>
> In contrast, the wave function and the Hamiltonian we'll use to implement the quantum solution are the result of transforming the canonical representation into a different one, more suitable for quantum computing. Such transformations preserve the physics of the system, but express them in a different form. We will discuss this in the next section of the kata.

The time-dependent Schrödinger equation we've seen above has solutions that correspond to _stationary states_, also known as _energy eigenstates_. For Hamiltonians that don't depend on time themselves, these states can be described by a simpler, time-independent form of Schrödinger equation:

$$\hat{H} \ket{\Psi_0} = E \ket{\Psi_0}$$

Here $E$ is a scalar, the energy of the system in the state $\ket{\Psi_0}$.

You will recognize this as the eigenvalue equation: $\hat{H}$ is a linear operator acting on vector $\ket{\Psi_0}$, and the result is the same vector, multiplied by the scalar $E$. The ground state is the energy eigenstate with the lowest energy, and the ground state energy is the corresponding eigenvalue.

This means that we can formulate the problem of finding the ground state energy as finding the eigenvalue of a certain operator associated with a certain eigenstate (or, alternatively, the lowest of all eigenvalues of this operator). And this is exactly the formulation of phase estimation problem!

## From molecule to quantum phase estimation inputs

To find the ground state energy of a molecule using quantum phase estimation algorithm, we need two inputs: the unitary that encodes the operator we're analyzing (the Hamiltonian of this molecule) and the eigenstate of this unitary that encodes the ground state. Note that both should be defined in terms that are suitable for implementing on a quantum computer rather than direct quantum-mechanical terms.

In this section, we'll take a look at the workflow that allows us to get from the quantum-mechanical description of the molecule to the inputs we can use with quantum phase estimation algorithm directly.

The first step is constructing the Hamiltonian of the molecule defined in terms of particles and forces acting between them. For many problems in chemistry, the Hamiltonian only considers the state of the electrons of the molecule, modeling the presence of nucleus via the electrostatic field generated by its electric charge (the Coulomb field). This model is called the Born-Oppenheimer approximation, and the resulting Hamiltonian is called the _fermionic Hamiltonian_, since electrons are fermions.

> Figuring out the Hamiltonian of a system and its ground state requires in-depth understanding of physics and chemistry of the interactions within the system. This step is usually done using classical chemistry tools, and their output serves as the input to the later steps.

Next, we need to convert the fermionic wave function and Hamiltonian from the previous step to a form described in quantum computing terms. This involves choosing a mapping of fermions onto qubits and transforming the wave function and the Hamiltonian in accordance to this mapping. Two commonly used techniques for doing this are called Jordan-Wigner transformation and Bravyi-Kitaev transformation. The result of such transformation is called _qubits Hamiltonian_.

> The details of these transformations are out of scope for this kata. For our task of finding the ground state energy of molecular hydrogen, we will use the Hamiltonian produced by the Bravyi-Kitaev transformation as described in the paper ["Scalable Quantum Simulation of Molecular Energies" by P.J.J.O'Malley et al.](https://arxiv.org/abs/1512.06860)

Can we find the eigenvalue of the qubits Hamiltonian and the corresponding representation of the ground state using quantum phase estimation? Not yet; we'll need to take one more step for that.

The qubits Hamiltonian is a Hermitian operator (that is, an operator that equals its conjugate transpose), which follows from the fact that it describes the energy of a physical system, even if represented in a different form than the original fermionic Hamiltonian. However, this Hamiltonian is not a unitary operator (that is, its inverse is not equal to its conjugate transpose), so we cannot implement it as part of the quantum phase estimation algorithm.

However, we can consider a different operator, related to the qubits Hamiltonian, called _unitary time evolution operator_. For a Hamiltonian $H$, this operator is defined as 

$$U(t) = e^{-itH}$$

> The unitary time evolution operator describes the evolution of a quantum state over time: if the initial state was $\ket{\psi(0)}$, the state after time $t$ will be $\ket{\psi(t)} = U(t) \ket{\psi(0)}$.

This operator is unitary, since the Hamiltonian $H$ is Hermitian. The eigenstates of the two operators are the same, so the ground state of the Hamiltonian is also an eigenstate of the unitary time evolution operator. The eigenvalues of the two operators are different, but they are related: an eigenvalue $E$ of the Hamiltonian corresponds to an eigenvalue $e^{-itE}$ of the unitary time evolution operator.

This means that we can finally use quantum phase estimation to solve our problem! If we implement the unitary time evolution operator generated by the qubits Hamiltonian and the corresponding ground state wave function, we will be able to convert the output of the phase estimation algorithm to the energy of the ground state.

## Demo: Is time evolution operator unitary?

Let's forget about the problem of analyzing a real Hamiltonian for a moment and take a look at a very simple example of a Hermitian matrix to check whether the corresponding time evolution operator will indeed be a unitary matrix.

Consider a matrix $A = \begin{bmatrix} 1 & 1 \\ 1 & -1 \end{bmatrix}$. It is Hermitian, since it is a symmetric matrix and all its elements are real numbers. However, it is not unitary: the inverse of $A$ $A^{-1} = \tfrac12 \begin{bmatrix} 1 & 1 \\ 1 & -1 \end{bmatrix}$. (Notice that $A$ is the matrix of the Hadamard gate without the $\frac1{\sqrt2}$ normalization coefficient.)

In the following demo, we'll use scipy library function `expm` to calculate matrix exponential $e^{-itA}$ for several values of $t$ and check whether the resulting matrix is unitary every time.

In [1]:
import numpy as np
from scipy.linalg import expm

a = np.array([[1, 1], [1, -1]])

# Calculate time evolution operator for several different evolution time values
for t in range(1, 5):
    exp_ita = expm(-1j * t * a)
    print(exp_ita)
    # Check whether the resulting matrix is unitary:
    # multiply it by its conjugate transpose and check whether the result is identity
    is_unitary = np.allclose(np.eye(2), exp_ita @ exp_ita.conjugate().T)
    # Check whether the resulting matrix is Hermitian:
    # compare it with its conjugate transpose
    is_hermitian = np.allclose(exp_ita, exp_ita.conjugate().T)
    print(f"Matrix expm(-itA) for {t=} is {'unitary' if is_unitary else 'NOT unitary'} and {'Hermitian' if is_hermitian else 'NOT Hermitian'}\n")

[[ 1.55943695e-01-0.698456j  0.00000000e+00-0.698456j]
 [-1.25285803e-17-0.698456j  1.55943695e-01+0.698456j]]
Matrix expm(-itA) for t=1 is unitary and NOT Hermitian

[[-9.51363128e-01-0.21783962j  0.00000000e+00-0.21783962j]
 [ 1.74777627e-16-0.21783962j -9.51363128e-01+0.21783962j]]
Matrix expm(-itA) for t=2 is unitary and NOT Hermitian

[[-4.52661857e-01+0.63051457j  8.13390491e-17+0.63051457j]
 [-1.23543213e-16+0.63051457j -4.52661857e-01-0.63051457j]]
Matrix expm(-itA) for t=3 is unitary and NOT Hermitian

[[ 8.10183603e-01+0.41448916j  7.67066993e-18+0.41448916j]
 [-3.28710293e-16+0.41448916j  8.10183603e-01-0.41448916j]]
Matrix expm(-itA) for t=4 is unitary and NOT Hermitian



## Demo: Eigenvalues and eigenvectors of a matrix and the corresponding time evolution operator

In the next demo, we'll explore the relationship between the eigenvectors and eigenvalues of a Hermitian matrix and the time evolution operator that corresponds to it. We'll use the same matrix as in the previous demo, $A = \begin{bmatrix} 1 & 1 \\ 1 & -1 \end{bmatrix}$, and the same range of values for evolution time $t$.

You can see that indeed, the eigenvectors of the time evolution operators remain the same (up to a global phase), and the eigenvalues change from real values $E$ to complex numbers $e^{-itE}$.

In [2]:
from cmath import exp, isclose
import numpy as np
from scipy.linalg import expm

a = np.array([[1, 1], [1, -1]])
# The matrix is Hermitian, so we can use the method eigh
a_eig_values, a_eig_vectors = np.linalg.eigh(a)
print("Hermitian matrix A:")
for ind in range(2):
    print(f"Eigenvalue {ind} = {a_eig_values[ind]}, eigenvector = {a_eig_vectors[:, ind]}")
    assert np.allclose(a @ a_eig_vectors[:, ind], a_eig_values[ind] * a_eig_vectors[:, ind])

for t in range(1, 5):
    exp_ita = expm(-1j * t * a)
    print(f"\nUnitary time evolution operator for {t=}")
    exp_eig_values, exp_eig_vectors = np.linalg.eig(exp_ita)
    # Reorder eigenvalues and eigenvectors so that the order matches that of the original matrix: 
    # eigenvector with different signs of the terms is first
    if exp_eig_vectors[0, 0] * exp_eig_vectors[1, 0] > 0:
        # Swap
        exp_eig_values = exp_eig_values[::-1]
        exp_eig_vectors[:, [0, 1]] = exp_eig_vectors[:, [1, 0]]
    for ind in range(2):
        print(f"Eigenvalue {ind} = {exp_eig_values[ind]}, eigenvector = {exp_eig_vectors[:, ind]}")
        assert np.allclose(a @ a_eig_vectors[:, ind], a_eig_values[ind] * a_eig_vectors[:, ind])
        assert isclose(exp_eig_values[ind], exp(-1j * t * a_eig_values[ind]))

Hermitian matrix A:
Eigenvalue 0 = -1.4142135623730951, eigenvector = [ 0.38268343 -0.92387953]
Eigenvalue 1 = 1.4142135623730951, eigenvector = [-0.92387953 -0.38268343]

Unitary time evolution operator for t=1
Eigenvalue 0 = (0.1559436947653745+0.9877659459927355j), eigenvector = [-0.38268343-5.55111512e-17j  0.92387953+0.00000000e+00j]
Eigenvalue 1 = (0.1559436947653745-0.9877659459927353j), eigenvector = [0.92387953+0.j 0.38268343-0.j]

Unitary time evolution operator for t=2
Eigenvalue 0 = (-0.9513631281258474+0.308071742363045j), eigenvector = [-0.38268343-1.21430643e-16j  0.92387953+0.00000000e+00j]
Eigenvalue 1 = (-0.9513631281258472-0.308071742363045j), eigenvector = [0.92387953+0.00000000e+00j 0.38268343+2.54918375e-16j]

Unitary time evolution operator for t=3
Eigenvalue 0 = (-0.4526618572923522-0.8916822544789359j), eigenvector = [-0.38268343+6.59194921e-17j  0.92387953+0.00000000e+00j]
Eigenvalue 1 = (-0.45266185729235264+0.8916822544789353j), eigenvector = [0.92387953+0.0

## Implementing unitary time evolution operator

Unitary time evolution operator $U(t)$ can be very complicated, and it's not obvious at first glance how to implement it in our quantum program. Ideally, we'd like to have a general approach to implementing it that does not depend on the exact form the Hamiltonian takes. Let's take a look at one such approach.

### Representing the Hamiltonian as a sum of Pauli matrices

Let's represent the qubits Hamiltonian as a linear combination of tensor products of Pauli matrices $I$, $X$, $Y$, $Z$:

$$H = \sum_{j = 0}^{m-1} c_j \bigotimes_{k=0}^{n_q-1} P_k^{(j)}$$

Here

* $m$ is the number of tensor product terms in the linear combination
* $c_j$ is the real coefficient of $j$-th tensor product term
* $n_q$ is the number of qubits the Hamiltonian acts on
* $P_k^{(j)}$ is the Pauli matrix that acts on qubit $k$ in the $j$-th tensor product term 

Such representation is always possible, because the four Pauli matrices form a basis for matrices of arbitrary sizes. You can represent any single-qubit matrix as a linear combination of Pauli matrices, any two-qubit matrix - as a linear combination of tensor products of pairs of Pauli matrices, and so on. The simplest example is the single-qubit matrices: you can always represent a $2 \times 2$ matrix as the following sum:

$$H = c_0 I + c_1 X + c_2 Y + c_3 Z$$

This is the same formula we've seen before, with $n_q = 1$, $m = 4$, $P_0^{(0)} = I, P_0^{(1)} = X, P_0^{(2)} = Y, P_0^{(3)} = Z$.

> Conveniently, the representation of the qubits Hamiltonian as a sum of Pauli tensor products is the direct result of applying the Jordan-Wigner or Bravyi-Kitaev transformation that we used to convert the fermionic Hamiltonian into the qubits Hamiltonian!

How does this representation help us?

### Trotter decomposition

*Trotter decomposition* is a method of approximating the exponential of a sum of terms for the case when these terms don't commute.

Consider, for example, the expression $e^{A+B}$, where $A$ and $B$ are both matrices. If $A$ and $B$ don't commute ($AB \neq BA$), we cannot just replace $e^{A+B}$ with $e^A e^B$ like we would do with scalars - these are different matrices. However, we can *approximate* $e^{A+B}$ with a product of multiple terms as follows:

$$e^{A+B} \approx \left( e^{\frac{A}{n}} e^{\frac{B}{n}} \right)^n$$

In the general case, when there are $m$ terms in the sum, the approximation looks as follows:

$$e^{A_0 + A_1 + ... + A_{m-1}} \approx \left( e^{\frac{A_0}{n}} e^{\frac{A_1}{n}} ... e^{\frac{A_{m-1}}{n}} \right)^n$$

The product of scaled exponents $e^{\frac{A_0}{n}} ... e^{\frac{A_{m-1}}{n}}$ is known as one *Trotter step*. As the number of Trotter steps $n$ grows, the approximation becomes more accurate (that is, the matrix produced by the product of Trotter steps becomes closer to the true matrix). However, increasing the value of $n$ also makes implementing the approximation more expensive in terms of the number of gates it uses.


### Susuki-Trotter decomposition

*Susuki-Trotter decomposition* is an alternative method of approximating the same exponential. It uses different formulas for different orders; the Trotter decomposition we saw above corresponds to the first order Susuki-Trotter decomposition (often referred to as "first order Trotterization"). We'll take a look at second order decomposition here; higher-order decompositions are out of scope of this kata.

> Even the state-of-the-art applications rarely consider decompositions beyond second order, so we're not missing much by omitting them in this kata!

For the two-term example $e^{A+B}$, the second order Trotterization looks as follows:

$$e^{A+B} \approx \left( e^{\frac{A}{2n}} e^{\frac{B}{n}} e^{\frac{A}{2n}} \right)^n$$

In the general case of an $m$-term sum, the approximation looks as follows:

$$e^{A_0 + A_1 + ... + A_{m-1}} \approx \left( e^{\frac{A_0}{2n}} ... e^{\frac{A_{m-2}}{2n}} e^{\frac{A_{m-1}}{n}} e^{\frac{A_{m-2}}{2n}} ... e^{\frac{A_0}{2n}} \right)^n$$

> You can think about these decompositions as follows: 
> * First order Trotterization scales individual terms by a factor of $n$ and goes through the terms in order, $\frac{A_0}{n}$, ..., $\frac{A_{m-1}}{n}$.
> * Second order Trotterization scales individual terms by a factor of $2n$ and goes through them first in order, $\frac{A_0}{2n}$, ..., $\frac{A_{m-1}}{2n}$, then in reverse order, $\frac{A_{m-1}}{2n}$, ..., $\frac{A_0}{2n}$. The last term in the first sequence and the first term and in second sequence is the same, $\frac{A_{m-1}}{2n}$, so we combine them into one term $\frac{A_{m-1}}{n}$.

## Demo: Approximating unitary time evolution operator

Let's revisit the Hermitian matrix we considered earlier, $A = \begin{bmatrix} 1 & 1 \\ 1 & -1 \end{bmatrix}$, to illustrate how Trotterization works to approximate unitary time evolution operator.

We can represent this matrix as a sum of two Pauli matrices as follows:

$$A = X + Z$$

The matrices $X$ and $Z$ don't commute: $[X, Z] = XZ - ZX = 2XZ \neq 0$. Thus, we know that $e^{-itX}e^{-itZ} \neq e^{-it(X+Z)}$.

In the following demo, we'll use two variants of Trotterization with varying number of Trotter steps $N$:

1. First order Trotter decomposition approximates $e^{-it(X+Z)}$ as follows:

$$e^{-it(X+Z)} \approx \prod_{k=1}^N e^{-i \frac{t}{N} X} e^{-i \frac{t}{N} Z}$$

2. Second order Suzuki-Trotter decomposition approximates $e^{-it(X+Z)}$ as follows:

$$e^{-it(X+Z)} \approx \prod_{k=1}^N e^{-i \frac{t}{2N} X} e^{-i \frac{t}{N} Z} e^{-i \frac{t}{2N} X}$$

In [3]:
import numpy as np
from scipy.linalg import expm

x = np.array([[0, 1], [1, 0]])
z = np.array([[1, 0], [0, -1]])
a = x + z
t = 2

exp_ita = expm(-1j * t * a)

print("|  n  | 1st order | 2nd order |")
for n in range(1, 51, 3):
    exp_itxn = expm(-1j * t * x / n)
    exp_itzn = expm(-1j * t * z / n)
    exp_itxn2 = expm(-1j * t * x / 2 / n)

    first_order_approximation = np.linalg.matrix_power(exp_itxn @ exp_itzn, n)
    first_order_norm_diff = np.linalg.norm(first_order_approximation - exp_ita)

    second_order_approximation = np.linalg.matrix_power(exp_itxn2 @ exp_itzn @ exp_itxn2, n)
    second_order_norm_diff = np.linalg.norm(second_order_approximation - exp_ita)

    print(f"| {n:2d}  |  {first_order_norm_diff:.5f}  |  {second_order_norm_diff:.5f}  |")

|  n  | 1st order | 2nd order |
|  1  |  2.30618  |  2.04852  |
|  4  |  0.19462  |  0.09236  |
|  7  |  0.09577  |  0.02912  |
| 10  |  0.06429  |  0.01414  |
| 13  |  0.04862  |  0.00834  |
| 16  |  0.03917  |  0.00550  |
| 19  |  0.03282  |  0.00389  |
| 22  |  0.02826  |  0.00290  |
| 25  |  0.02482  |  0.00225  |
| 28  |  0.02213  |  0.00179  |
| 31  |  0.01997  |  0.00146  |
| 34  |  0.01819  |  0.00121  |
| 37  |  0.01671  |  0.00103  |
| 40  |  0.01545  |  0.00088  |
| 43  |  0.01436  |  0.00076  |
| 46  |  0.01342  |  0.00066  |
| 49  |  0.01260  |  0.00058  |


You can see that, indeed, increasing $N$ reduces the error of both approximations. Furthermore, second order decomposition reduces the error much faster than first order decomposition.

### Summary: Approximating unitary time evolution operator

To summarize, here are the steps we'll need to approximate the unitary time evolution operator for a given Hamiltonian:

1. Obtain the representation of the qubits Hamiltonian as a sum of Pauli tensor products.
2. Learn to implement unitaries of the form $\exp(i \alpha P_0 \otimes P_1 \otimes ... P_{nq-1})$, where $\alpha$ is a real number and each $P_k$ is a Pauli matrix. These unitaries are called *Pauli product rotations* (PPRs).
3. Use first or second order Trotterization to approximate the unitary time evolution operator as a sequence of PPRs with appropriate coefficients.

Finally, we'll use this approximation as one of the inputs to the quantum phase estimation algorithm!

> What about the second input to phase estimation, the eigenstate? The problem of preparing an arbitrary quantum state is comparatively much more straightforward (at least conceptually; doing it efficiently is another question!) The amplitudes that describe the eigenstate of the unitary time evolution operator are obtained via the same transformation we use to convert fermionic Hamiltonian to qubits Hamiltonian (remember that the eigenstate we're interested in is the ground state of the Hamiltonian).
>
> It is possible that this process won't give us the precise eigenstate, but rather a very good approximation for it. Quantum phase estimation algorithm works for superpositions of eigenstates as well, producing a superposition of corresponding eigenvalues instead of just one. If the state we prepare is very close to the true eigenstate, we'll get the eigenvalue we're looking for most of the time!

## Part I conclusion

Congratulations! In the first part of this kata, you've learned the basic flow of using quantum computing for solving the problem of finding ground state energy of a molecule.
With this in mind, let's see how this process looks when implemented as a quantum program in [part II](./GroundStateEnergyCode.ipynb)!